[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/51_polygon_clip.ipynb)

# 🔴 Hard: Polygon Clip (Sutherland-Hodgman)

Clip a `subject` polygon against a convex `clip` polygon and return the intersection polygon. Returns an empty array `(0, 2)` if there is no intersection.

### Signature
```python
def polygon_clip(subject: np.ndarray, clip: np.ndarray) -> np.ndarray:
    # subject: (N, 2) float — any polygon
    # clip:    (M, 2) float — convex polygon in CCW order
    # returns: (K, 2) float — clipped polygon, or np.zeros((0,2))
```

### Rules
- Implement **Sutherland-Hodgman** polygon clipping
- The `clip` polygon must be convex and in CCW order
- "Inside" = to the left of (or on) each directed clip edge

### Example
```
subject = [[0,0],[4,0],[4,4],[0,4]]  # big square
clip    = [[1,1],[3,1],[3,3],[1,3]]  # smaller square clip
output: vertices of the smaller square (area=4)
```

> **Reduction step (say this before coding):** For each directed edge of `clip`, keep only the parts of `subject` to its left — iterate edge by edge, each iteration shrinking `output` by one half-plane.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def polygon_clip(subject, clip):
    # subject: (N, 2) — any polygon
    # clip:    (M, 2) — convex clip polygon, CCW order
    # returns: (K, 2) clipped polygon, or np.zeros((0,2)) if empty
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
# Big square clipped by small square → small square
big  = np.array([[0.,0.],[4.,0.],[4.,4.],[0.,4.]])
clip = np.array([[1.,1.],[3.,1.],[3.,3.],[1.,3.]])
result = polygon_clip(big, clip)
print("Clipped polygon:", result)
print("Num vertices:", len(result))

# Helper to compute area
def area(p):
    p=np.array(p,dtype=float); x,y=p[:,0],p[:,1]
    return 0.5*abs(np.sum(x*np.roll(y,-1)-np.roll(x,-1)*y))
print("Area:", area(result), "  expected: 4.0")

In [ ]:
# ✅ Inline test suite
import numpy as np, time

def area(p):
    p=np.array(p,dtype=float); x,y=p[:,0],p[:,1]
    return 0.5*abs(np.sum(x*np.roll(y,-1)-np.roll(x,-1)*y))

# ── Test 1: clip inside bigger square ─────────────────────────────────────
big=np.array([[0.,0.],[4.,0.],[4.,4.],[0.,4.]])
clip=np.array([[1.,1.],[3.,1.],[3.,3.],[1.,3.]])
r=polygon_clip(big,clip); r=np.array(r,dtype=float)
assert len(r)>=3, f"Expected polygon, got {len(r)} pts"
assert abs(area(r)-4.)<1e-6, f"Area: {area(r):.6f} expected 4.0"
print("Test 1 passed: inner square clip")

# ── Test 2: subject fully inside clip ──────────────────────────────────────
subj=np.array([[1.,1.],[2.,1.],[2.,2.],[1.,2.]])
clip2=np.array([[0.,0.],[4.,0.],[4.,4.],[0.,4.]])
r2=polygon_clip(subj,clip2); r2=np.array(r2,dtype=float)
assert abs(area(r2)-1.)<1e-6, f"Fully inside: area {area(r2):.6f} expected 1.0"
print("Test 2 passed: subject fully inside clip")

# ── Test 3: no intersection ────────────────────────────────────────────────
far=np.array([[10.,10.],[12.,10.],[12.,12.],[10.,12.]])
r3=np.array(polygon_clip(far, clip2))
assert len(r3)==0, f"No intersection expected, got {len(r3)} pts"
print("Test 3 passed: no intersection → empty")

# ── Test 4: partial overlap — triangle clip ────────────────────────────────
sq2=np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
tri_clip=np.array([[0.,0.],[2.,0.],[0.,2.]])
r4=np.array(polygon_clip(sq2, tri_clip),dtype=float)
assert len(r4)>=3
assert abs(area(r4)-2.)<1e-6, f"Triangle clip area: {area(r4):.6f} expected 2.0"
print("Test 4 passed: triangle clip of square")

# ── Test 5: large vertices ─────────────────────────────────────────────────
n_s,n_c=80,20
a_s=np.linspace(0,2*np.pi,n_s,endpoint=False)
a_c=np.linspace(0,2*np.pi,n_c,endpoint=False)
subj5=np.stack([2*np.cos(a_s),2*np.sin(a_s)],axis=1)
clip5=np.stack([1.5*np.cos(a_c),1.5*np.sin(a_c)],axis=1)
t0=time.time(); r5=np.array(polygon_clip(subj5,clip5),dtype=float); elapsed=time.time()-t0
assert len(r5)>=3
assert elapsed<1.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 80-gon clipped by 20-gon ({elapsed:.3f}s)")

print("\nAll tests passed!")